# 🚀 Дообучение ASR модели

## 📦 Импорт библиотек

In [1]:
# Базовые библиотеки
import numpy as np
import evaluate
import os
os.environ["DATASETS_DISABLE_TORCHCODEC"] = "1"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# Работа с набором данных
import librosa
from datasets import load_dataset, Features, Value
from dataclasses import dataclass
from typing import Any, Dict, List, Union

# Обучение
import torch
from transformers import (
    WhisperForConditionalGeneration, 
    WhisperProcessor, 
    Trainer, 
    TrainingArguments,
)

C:\Users\Vlad\anaconda3\envs\ml\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 🔧 Конфигурация

In [2]:
# Назвние модели
model_name = "openai/whisper-small"
output_dir = "./whisper-finetuned-ru"
# Путь к набору данных
audio_dir = "cv-corpus-24.0-2025-12-05/ru/clips"

## 📥 Импорт набора данных

In [3]:
# Ограничиваем набор данных для ускоренного обучения
MAX_TRAIN_SAMPLES = 3000
MAX_EVAL_SAMPLES = 500
NUM_EPOCHS = 1    

# Найстройка типов данных для столбцов
features = Features({
    "client_id": Value("string"),
    "path": Value("string"),
    "sentence_id": Value("string"),
    "sentence": Value("string"),
    "sentence_domain": Value("string"),
    "up_votes": Value("string"),
    "down_votes": Value("string"),
    "age": Value("string"),         
    "gender": Value("string"),         
    "accents": Value("string"),        
    "variant": Value("string"),    
    "locale": Value("string"),
    "segment": Value("string"),         
})

# Импорт набора данных (train, test, validation)
dataset = load_dataset(
    "csv",
    data_files={
        "train": "cv-corpus-24.0-2025-12-05/ru/train.tsv",
        "validation": "cv-corpus-24.0-2025-12-05/ru/dev.tsv",
        "test": "cv-corpus-24.0-2025-12-05/ru/test.tsv",
    },
    delimiter="\t",
    features=features
)

dataset["train"] = dataset["train"].select(range(min(MAX_TRAIN_SAMPLES, len(dataset["train"]))))
dataset["validation"] = dataset["validation"].select(range(min(MAX_EVAL_SAMPLES, len(dataset["validation"]))))

# Путь до аудио файлов
def add_audio_path(example, audio_dir):
    import os
    example["audio_path"] = os.path.join(audio_dir, example["path"])
    return example

dataset = dataset.map(
    add_audio_path, 
    fn_kwargs={"audio_dir": audio_dir},
    num_proc=4
)

Map (num_proc=4): 100%|██████████████████████████████████████████████████████| 500/500 [00:02<00:00, 230.63 examples/s]


In [4]:
processor = WhisperProcessor.from_pretrained(model_name, language="ru", task="transcribe")

## 🧹 Обработка данных

In [5]:
# Обработка аудио файлов
def prepare_dataset(batch, processor):
    import os
    import librosa
    
    try:
        # 16Khz
        audio_array, sr = librosa.load(batch["audio_path"], sr=16000)
        
        batch["input_features"] = processor.feature_extractor(
            audio_array, 
            sampling_rate=sr
        ).input_features[0]

        # Токенизация
        batch["labels"] = processor.tokenizer(batch["sentence"]).input_ids
        
        return batch
    except Exception as e:
        print(f"Error loading {batch['audio_path']}: {e}")
        batch["input_features"] = None
        batch["labels"] = None
        return batch

dataset = dataset.map(
    prepare_dataset, 
    fn_kwargs={"processor": processor},
    num_proc=4, 
    desc="Processing"
)

Processing (num_proc=4): 100%|███████████████████████████████████████████| 10261/10261 [01:01<00:00, 168.06 examples/s]


In [6]:
# Фильтрация ошибочных примеров
dataset = dataset.filter(
    lambda x: x["input_features"] is not None, 
    num_proc=4
)

Filter (num_proc=4): 100%|████████████████████████████████████████████████| 10261/10261 [03:10<00:00, 53.80 examples/s]


In [7]:
# Удаление лишних колонок
dataset = dataset.remove_columns(
    [col for col in dataset["train"].column_names 
     if col not in ["input_features", "labels"]]
)

print(f"✅ Train: {len(dataset['train'])}, Validation: {len(dataset['validation'])}")

✅ Train: 3000, Validation: 500


In [8]:
# Подготовка данных
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]
        
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), 
            -100
        )
        
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]
        
        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

In [18]:
# Параметры обучения
import accelerate
training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=1e-5,
    warmup_steps=50,
    num_train_epochs=NUM_EPOCHS,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    fp16=True,
    remove_unused_columns=False,
    label_names=["labels"],
    push_to_hub=False,
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    report_to="none",
)

ImportError: Using the `Trainer` with `PyTorch` requires `accelerate>=1.1.0`: Please run `pip install transformers[torch]` or `pip install 'accelerate>=1.1.0'`

In [14]:
# Создание модели
model = WhisperForConditionalGeneration.from_pretrained(model_name)
model.generation_config.language = "ru"
model.generation_config.task = "transcribe"
model.generation_config.max_length = 225
model.config.forced_decoder_ids = None

KeyboardInterrupt: 

In [ ]:
# Метрики оценки
wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    
    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    
    wer = 100 * wer_metric.compute(predictions=pred_str, references=label_str)
    
    return {"wer": wer}

In [ ]:
# Обучение модели
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    tokenizer=processor.feature_extractor,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("🚀 Starting training...")
trainer.train()

# Сохранение
trainer.save_model(f"{output_dir}-final")
processor.save_pretrained(f"{output_dir}-final")
print(f"✅ Model saved to {output_dir}-final")